# Download Sentinel-1 radar images

Downloads Sentinel-1 radar images (VV and VH polarisation, in dB, 10 m pixels) from the
**Copernicus Data Space Ecosystem** into a local archive of GeoTIFF files.

For every area in `S1_AREAS` (in `.env`) it looks back `S1_LOOKBACK_DAYS` days, finds the satellite
passes that cover the area well, and downloads the newest `S1_MAX_PASSES` of them:

```
<S1_DATA_DIR>/
    index.csv                       one row per downloaded pass
    <area>/<YYYY-MM-DD_HHMM>/
        VV_dB.tif, VH_dB.tif        GeoTIFF, float32 dB, EPSG:32631 (UTM 31N), 10 m, nodata -9999
        quicklook.png               grey preview (ships are bright dots)
        scene.json                  scene ids, acquisition time, georeference, coverage
```

Passes already in the archive are never downloaded twice.

## Settings
Credentials and defaults live in `.env`. The values below only override them for this run;
`None` means "use `.env`".

In [ ]:
ENV_FILE = None      # path to .env; None = look in this folder (and its parent folders)
AREAS = None         # e.g. ["rotterdam"]; None = every area in S1_AREAS
DATE = None          # e.g. "2026-10-25" = only passes on that UTC day; None = the last S1_LOOKBACK_DAYS days
MAX_PASSES = None    # newest passes per area to keep; None = S1_MAX_PASSES
LIST_ONLY = False    # True = only list available passes and their cost, download nothing
FORCE = False        # True = download again even if the pass is already in the archive

## Imports and constants

In [ ]:
import csv
import json
import math
import os
import time
from datetime import datetime, timedelta, timezone
from io import BytesIO

import numpy as np
import requests
import tifffile
from PIL import Image

TOKEN_URL = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
SH = "https://sh.dataspace.copernicus.eu"
PROCESS_URL = SH + "/api/v1/process"
CATALOG_URLS = [SH + "/api/v1/catalog/1.0.0/search", SH + "/catalog/v1/search"]

UTM_ZONE = 31
EPSG = 32600 + UTM_ZONE
CRS_URL = f"http://www.opengis.net/def/crs/EPSG/0/{EPSG}"
PIXEL = 10.0          # metres, same as xView3
BLOCK = 2048          # px per request side (the API allows at most 2500)
NODATA = -9999.0
DEFAULTS = {"S1_DATA_DIR": os.path.join("data", "sentinel1"),
            "S1_AREAS": "rotterdam:3.55,51.92,3.95,52.12",
            "S1_LOOKBACK_DAYS": "10", "S1_MAX_PASSES": "1", "S1_MIN_COVERAGE": "0.5"}

# Runs on the Copernicus servers for every pixel: radar backscatter -> dB, no-data -> -9999
EVALSCRIPT = f"""//VERSION=3
function setup() {{
  return {{ input: [{{ bands: ["VV", "VH", "dataMask"] }}],
           output: {{ bands: 2, sampleType: "FLOAT32" }} }};
}}
function evaluatePixel(s) {{
  if (s.dataMask == 0) return [{NODATA}, {NODATA}];
  return [10 * Math.log(Math.max(s.VV, 1e-10)) / Math.LN10,
          10 * Math.log(Math.max(s.VH, 1e-10)) / Math.LN10];
}}"""


class StopRun(Exception):
    """A problem that needs your attention (missing credentials, no data, ...)."""

## Reading `.env`

In [ ]:

def find_env(explicit=None):
    """The .env file: explicit path, else in the current folder or up to two parent folders."""
    if explicit:
        return explicit
    folder = os.getcwd()
    for _ in range(3):
        p = os.path.join(folder, ".env")
        if os.path.isfile(p):
            return p
        folder = os.path.dirname(folder)
    return None


def load_env(path):
    """Minimal .env parser: KEY=VALUE lines, # comments, optional quotes and 'export '."""
    values = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            if line.startswith("export "):
                line = line[7:]
            key, val = line.split("=", 1)
            val = val.strip()
            if len(val) >= 2 and val[0] == val[-1] and val[0] in "\"'":
                val = val[1:-1]
            elif " #" in val:
                val = val.split(" #", 1)[0].strip()
            values[key.strip()] = val
    return values


def settings(env_file=None, need=()):
    """Read .env."""
    path = find_env(env_file)
    env = load_env(path)
    missing = [k for k in need if not env.get(k)]
    if missing:
        raise StopRun(f"{path} is missing a value for: {', '.join(missing)}")
    for k, v in DEFAULTS.items():
        if not env.get(k):
            env[k] = v
    env["_path"] = path
    return env


def parse_areas(text):
    """'name:lon_min,lat_min,lon_max,lat_max;name2:...' -> {name: [lon_min, lat_min, lon_max, lat_max]}"""
    areas = {}
    for part in text.split(";"):
        part = part.strip()
        if not part:
            continue
        try:
            name, coords = part.split(":", 1)
            bbox = [float(v) for v in coords.split(",")]
            assert len(bbox) == 4 and bbox[0] < bbox[2] and bbox[1] < bbox[3]
        except Exception:
            raise StopRun(f"Bad S1_AREAS entry {part!r}; expected name:lon_min,lat_min,lon_max,lat_max")
        safe = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in name.strip().lower())
        areas[safe] = bbox
    if not areas:
        raise StopRun("S1_AREAS is empty.")
    return areas

## Coordinates
Images are stored on a 10 m grid in UTM zone 31N, the standard map projection for the Netherlands.
These functions convert between latitude/longitude and UTM metres, and estimate how much of an
area a satellite pass covers (from the scene outlines in the catalogue) before downloading anything.

In [ ]:
_A, _F, _K0 = 6378137.0, 1 / 298.257223563, 0.9996
_N = _F / (2 - _F)
_AA = _A / (1 + _N) * (1 + _N ** 2 / 4 + _N ** 4 / 64)
_ALPHA = (_N / 2 - 2 * _N ** 2 / 3 + 5 * _N ** 3 / 16, 13 * _N ** 2 / 48 - 3 * _N ** 3 / 5, 61 * _N ** 3 / 240)
_BETA = (_N / 2 - 2 * _N ** 2 / 3 + 37 * _N ** 3 / 96, _N ** 2 / 48 + _N ** 3 / 15, 17 * _N ** 3 / 480)
_DELTA = (2 * _N - 2 * _N ** 2 / 3 - 2 * _N ** 3, 7 * _N ** 2 / 3 - 8 * _N ** 3 / 5, 56 * _N ** 3 / 15)


def latlon_to_utm(lat, lon, zone=UTM_ZONE):
    lam0 = math.radians(zone * 6 - 183)
    phi, lam = math.radians(lat), math.radians(lon) - lam0
    e = math.sqrt(_F * (2 - _F))
    t = math.sinh(math.atanh(math.sin(phi)) - e * math.atanh(e * math.sin(phi)))
    xi, eta = math.atan2(t, math.cos(lam)), math.atanh(math.sin(lam) / math.sqrt(1 + t * t))
    x = eta + sum(a * math.cos(2 * j * xi) * math.sinh(2 * j * eta) for j, a in enumerate(_ALPHA, 1))
    y = xi + sum(a * math.sin(2 * j * xi) * math.cosh(2 * j * eta) for j, a in enumerate(_ALPHA, 1))
    return 500000 + _K0 * _AA * x, _K0 * _AA * y


def utm_to_latlon(easting, northing, zone=UTM_ZONE):
    xi = northing / (_K0 * _AA)
    eta = (easting - 500000) / (_K0 * _AA)
    xi_ = xi - sum(b * math.sin(2 * j * xi) * math.cosh(2 * j * eta) for j, b in enumerate(_BETA, 1))
    eta_ = eta - sum(b * math.cos(2 * j * xi) * math.sinh(2 * j * eta) for j, b in enumerate(_BETA, 1))
    chi = math.asin(math.sin(xi_) / math.cosh(eta_))
    phi = chi + sum(d * math.sin(2 * j * chi) for j, d in enumerate(_DELTA, 1))
    lam = math.atan2(math.sinh(eta_), math.cos(xi_))
    return math.degrees(phi), zone * 6 - 183 + math.degrees(lam)


def utm_grid(bbox):
    """Snap a lon/lat bbox to the 10 m UTM grid -> (x0, y0_top, width_px, height_px)."""
    corners = [latlon_to_utm(lat, lon) for lon in (bbox[0], bbox[2]) for lat in (bbox[1], bbox[3])]
    xs, ys = [c[0] for c in corners], [c[1] for c in corners]
    x0, x1 = math.floor(min(xs) / PIXEL) * PIXEL, math.ceil(max(xs) / PIXEL) * PIXEL
    y0, y1 = math.floor(min(ys) / PIXEL) * PIXEL, math.ceil(max(ys) / PIXEL) * PIXEL
    return x0, y1, int(round((x1 - x0) / PIXEL)), int(round((y1 - y0) / PIXEL))


def _in_ring(lon, lat, ring):
    inside, j = False, len(ring) - 1
    for i in range(len(ring)):
        xi, yi, xj, yj = ring[i][0], ring[i][1], ring[j][0], ring[j][1]
        if (yi > lat) != (yj > lat) and lon < (xj - xi) * (lat - yi) / ((yj - yi) or 1e-12) + xi:
            inside = not inside
        j = i
    return inside


def _rings(geom):
    if not geom:
        return []
    if geom.get("type") == "Polygon":
        return [geom["coordinates"][0]]
    if geom.get("type") == "MultiPolygon":
        return [poly[0] for poly in geom["coordinates"]]
    return []


def estimate_coverage(bbox, geoms, n=15):
    """Share of a grid of points in bbox that falls inside any scene outline (None if unknown)."""
    rings = [r for g in geoms for r in _rings(g)]
    if not rings:
        return None
    hit = sum(any(_in_ring(bbox[0] + (bbox[2] - bbox[0]) * (i + .5) / n,
                           bbox[1] + (bbox[3] - bbox[1]) * (j + .5) / n, r) for r in rings)
              for i in range(n) for j in range(n))
    return hit / (n * n)

## Copernicus client
Logs in with your OAuth client, searches the catalogue for Sentinel-1 scenes, and asks the
Process API for blocks of the image already converted to dB (the server does the radar processing).

In [ ]:
def iso(dt):
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def parse_iso(s):
    return datetime.fromisoformat(s.replace("Z", "+00:00"))


class CDSE:
    def __init__(self, cid, secret):
        self.cid, self.secret = cid, secret
        self.s = requests.Session()
        self.token, self.exp = None, 0
        self.catalog_url = None

    def _auth(self):
        if self.token and time.time() < self.exp - 60:
            return
        r = self.s.post(TOKEN_URL, data={"grant_type": "client_credentials",
                                         "client_id": self.cid, "client_secret": self.secret}, timeout=60)
        if not r.ok:
            raise StopRun(f"Login failed ({r.status_code}). Check CDSE_CLIENT_ID / CDSE_CLIENT_SECRET in .env\n"
                          f"{r.text[:400]}")
        d = r.json()
        self.token, self.exp = d["access_token"], time.time() + int(d.get("expires_in", 300))

    def post(self, url, body, accept=None):
        r = None
        for attempt in range(8):
            self._auth()
            h = {"Authorization": f"Bearer {self.token}", "Content-Type": "application/json"}
            if accept:
                h["Accept"] = accept
            r = self.s.post(url, json=body, headers=h, timeout=300)
            if r.status_code == 401 and attempt == 0:
                self.token = None
                continue
            if r.status_code == 429 or r.status_code >= 500:
                time.sleep(min(5 * (attempt + 1), 60))
                continue
            return r
        return r

    def search(self, bbox, t0, t1):
        """Sentinel-1 GRD IW scenes intersecting bbox in [t0, t1], newest first (with outlines)."""
        body = {"bbox": bbox, "datetime": f"{iso(t0)}/{iso(t1)}", "collections": ["sentinel-1-grd"], "limit": 100}
        urls = [self.catalog_url] if self.catalog_url else CATALOG_URLS
        for url in urls:
            out, nxt = [], None
            while True:
                r = self.post(url, dict(body, **({"next": nxt} if nxt is not None else {})))
                if r.status_code == 404 and not self.catalog_url:
                    break
                if not r.ok:
                    raise StopRun(f"Catalog search failed ({r.status_code}):\n{r.text[:600]}")
                self.catalog_url = url
                d = r.json()
                for f in d.get("features") or d.get("items") or []:
                    p = f.get("properties", {})
                    if p.get("datetime") and p.get("sar:instrument_mode") in (None, "IW"):
                        out.append({"id": f.get("id"), "datetime": p["datetime"],
                                    "orbit": p.get("sat:orbit_state"), "geometry": f.get("geometry")})
                nxt = (d.get("context") or {}).get("next")
                if nxt is None:
                    break
            if self.catalog_url:
                return sorted(out, key=lambda s: s["datetime"], reverse=True)
        raise StopRun("Could not reach the Copernicus catalogue.")

    def block(self, x0, y0, w, h, t0, t1):
        """w x h px block with upper-left UTM corner (x0, y0) -> array (h, w, 2) float32 dB."""
        body = {
            "input": {
                "bounds": {"bbox": [x0, y0 - h * PIXEL, x0 + w * PIXEL, y0], "properties": {"crs": CRS_URL}},
                "data": [{
                    "type": "sentinel-1-grd",
                    "dataFilter": {"timeRange": {"from": iso(t0), "to": iso(t1)},
                                   "acquisitionMode": "IW", "polarization": "DV",
                                   "resolution": "HIGH", "mosaickingOrder": "mostRecent"},
                    "processing": {"backCoeff": "SIGMA0_ELLIPSOID", "orthorectify": False},
                }],
            },
            "output": {"width": w, "height": h,
                       "responses": [{"identifier": "default", "format": {"type": "image/tiff"}}]},
            "evalscript": EVALSCRIPT,
        }
        r = self.post(PROCESS_URL, body, accept="image/tiff")
        if not r.ok:
            raise StopRun(f"Image request failed ({r.status_code}):\n{r.text[:600]}")
        try:
            arr = tifffile.imread(BytesIO(r.content)).astype(np.float32)
        except Exception as e:
            raise StopRun(f"Could not read the returned TIFF ({e}). Install imagecodecs: pip install imagecodecs")
        if arr.ndim == 3 and arr.shape[0] == 2 and arr.shape[-1] != 2:
            arr = np.moveaxis(arr, 0, -1)
        if arr.shape[:2] != (h, w):
            raise StopRun(f"Unexpected block shape {arr.shape}, wanted {(h, w, 2)}")
        return arr

## Saving
GeoTIFFs get proper map tags (QGIS, GDAL and rasterio read them directly); `index.csv` keeps track
of everything in the archive.

In [ ]:
def write_geotiff(path, arr, x0, y0):
    data = np.where(np.isfinite(arr), arr, NODATA).astype(np.float32)
    geokeys = [1, 1, 0, 3,           # version, revision, minor, number of keys
               1024, 0, 1, 1,        # GTModelTypeGeoKey = projected
               1025, 0, 1, 1,        # GTRasterTypeGeoKey = PixelIsArea
               3072, 0, 1, EPSG]     # ProjectedCSTypeGeoKey = EPSG:32631
    tifffile.imwrite(path, data, extratags=[
        (33550, "d", 3, (PIXEL, PIXEL, 0.0), True),               # ModelPixelScaleTag
        (33922, "d", 6, (0.0, 0.0, 0.0, x0, y0, 0.0), True),      # ModelTiepointTag (upper-left corner)
        (34735, "H", len(geokeys), geokeys, True),                # GeoKeyDirectoryTag
        (42113, "s", 0, f"{NODATA:g}", True),                     # GDAL_NODATA
    ])


def update_index(data_dir, entry):
    path = os.path.join(data_dir, "index.csv")
    rows = []
    if os.path.exists(path):
        with open(path, newline="", encoding="utf-8") as f:
            rows = [r for r in csv.DictReader(f) if r.get("folder") != entry["folder"]]
    rows.append(entry)
    rows.sort(key=lambda r: (r["area"], r["acquired_utc"]))
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(entry.keys()))
        w.writeheader()
        w.writerows(rows)


def group_passes(scenes):
    """Slices of one overpass are seconds apart: group them into passes, newest first."""
    passes = []
    for sc in sorted(scenes, key=lambda s: s["datetime"]):
        t = parse_iso(sc["datetime"])
        if passes and (t - passes[-1]["t_last"]).total_seconds() < 600:
            passes[-1]["scenes"].append(sc)
            passes[-1]["t_last"] = t
        else:
            passes.append({"t": t, "t_last": t, "scenes": [sc]})
    return passes[::-1]

## Download

In [ ]:
def download(env_file=None, areas=None, date=None, max_passes=None, list_only=False, force=False):
    env = settings(env_file, need=("CDSE_CLIENT_ID", "CDSE_CLIENT_SECRET"))
    data_dir = os.path.abspath(env["S1_DATA_DIR"])
    all_areas = parse_areas(env["S1_AREAS"])
    if areas:
        unknown = [a for a in areas if a not in all_areas]
        if unknown:
            raise StopRun(f"Unknown area(s) {unknown}; S1_AREAS has: {', '.join(all_areas)}")
        all_areas = {a: all_areas[a] for a in areas}
    max_passes = max_passes or int(env["S1_MAX_PASSES"])
    min_cov = float(env["S1_MIN_COVERAGE"])
    if date:
        t0 = datetime.strptime(date, "%Y-%m-%d").replace(tzinfo=timezone.utc)
        t1 = t0 + timedelta(days=1)
    else:
        t1 = datetime.now(timezone.utc)
        t0 = t1 - timedelta(days=int(env["S1_LOOKBACK_DAYS"]))
    print(f"Settings: {env['_path']}\nArchive:  {data_dir}\nWindow:   {iso(t0)} .. {iso(t1)}")

    api = CDSE(env["CDSE_CLIENT_ID"], env["CDSE_CLIENT_SECRET"])
    saved, reused, total_pu = [], 0, 0.0
    for area, bbox in all_areas.items():
        x0, y0, W, H = utm_grid(bbox)
        pu = W * H / 512 ** 2 * (2 / 3) * 2      # rough estimate: area x 2/3 (two bands) x 2 (float32)
        print(f"\n=== {area}  {bbox}  ({W} x {H} px, {W / 100:.1f} x {H / 100:.1f} km, ~{pu:.0f} PU per pass)")
        passes = group_passes(api.search(bbox, t0, t1))
        if not passes:
            print("  no Sentinel-1 passes in this window")
            continue
        done = 0
        for ps in passes:
            t = ps["t"]
            cov_est = estimate_coverage(bbox, [s["geometry"] for s in ps["scenes"]])
            folder = os.path.join(data_dir, area, t.strftime("%Y-%m-%d_%H%M"))
            in_archive = os.path.exists(os.path.join(folder, "scene.json"))
            label = (f"  {t:%Y-%m-%d %H:%M} UTC  {ps['scenes'][0].get('orbit') or '':<10} "
                     f"coverage ~{'?' if cov_est is None else f'{cov_est:.0%}':>4}  "
                     f"{len(ps['scenes'])} scene(s){'  [in archive]' if in_archive else ''}")
            if list_only:
                print(label)
                continue
            if done >= max_passes:
                break
            if cov_est is not None and cov_est < min_cov:
                print(label + "  -> skipped (too little coverage)")
                continue
            if in_archive and not force:
                print(label + "  -> already downloaded")
                done += 1
                reused += 1
                continue

            print(label + "  -> downloading")
            ts = [parse_iso(s["datetime"]) for s in ps["scenes"]]
            w0, w1 = min(ts) - timedelta(minutes=1), max(ts) + timedelta(minutes=2)
            vv = np.full((H, W), np.nan, np.float32)
            vh = np.full((H, W), np.nan, np.float32)
            blocks = [(r, c) for r in range(0, H, BLOCK) for c in range(0, W, BLOCK)]
            for i, (r, c) in enumerate(blocks, 1):
                h, w = min(BLOCK, H - r), min(BLOCK, W - c)
                print(f"     block {i}/{len(blocks)} ({w} x {h} px)")
                arr = api.block(x0 + c * PIXEL, y0 - r * PIXEL, w, h, w0, w1)
                arr[arr <= NODATA + 1] = np.nan
                vv[r:r + h, c:c + w], vh[r:r + h, c:c + w] = arr[..., 0], arr[..., 1]
            total_pu += pu
            valid = float(np.isfinite(vv).mean())
            if valid < min_cov:
                print(f"     only {valid:.0%} of the area has data -> not saved")
                continue

            os.makedirs(folder, exist_ok=True)
            write_geotiff(os.path.join(folder, "VV_dB.tif"), vv, x0, y0)
            write_geotiff(os.path.join(folder, "VH_dB.tif"), vh, x0, y0)
            ql = np.where(np.isfinite(vv), np.clip((vv + 30) / 40, 0, 1) * 254 + 1, 0).astype(np.uint8)
            im = Image.fromarray(ql)
            im.thumbnail((1600, 1600))
            im.save(os.path.join(folder, "quicklook.png"))
            sea = vv[np.isfinite(vv)]
            meta = {
                "area": area, "bbox_lonlat": bbox, "acquired_utc": iso(t),
                "scenes": [{k: s[k] for k in ("id", "datetime", "orbit")} for s in ps["scenes"]],
                "crs": f"EPSG:{EPSG}", "pixel_m": PIXEL, "utm_upper_left": [x0, y0], "size_px": [W, H],
                "nodata": NODATA, "units": "dB (10*log10 of sigma0)", "coverage": round(valid, 3),
                "processing": "Sentinel Hub sentinel-1-grd, IW, DV, HIGH, sigma0 ellipsoid, not orthorectified, "
                              "no speckle filter",
                "stats": {"VV_median_dB": round(float(np.median(sea)), 2),
                          "VH_median_dB": round(float(np.nanmedian(vh)), 2)},
                "downloaded_utc": iso(datetime.now(timezone.utc)),
            }
            with open(os.path.join(folder, "scene.json"), "w") as f:
                json.dump(meta, f, indent=2)
            update_index(data_dir, {
                "area": area, "acquired_utc": iso(t),
                "folder": os.path.relpath(folder, data_dir).replace("\\", "/"),
                "scenes": ";".join(s["id"] for s in ps["scenes"]), "coverage": round(valid, 3),
                "bbox": " ".join(str(v) for v in bbox),
            })
            print(f"     saved {folder}  (coverage {valid:.0%}, sea median VV {meta['stats']['VV_median_dB']} dB)")
            done += 1
            saved.append(folder)

    if list_only:
        print("\n(list only; nothing downloaded)")
    else:
        print(f"\nDone: {len(saved)} pass(es) downloaded (~{total_pu:.0f} PU), {reused} already in the archive.")
        print(f"Archive: {data_dir}   Next: 02_s1_to_tiles.ipynb")
    return {"data_dir": data_dir, "saved": saved}

## Run

In [ ]:
try:
    result = download(ENV_FILE, AREAS, DATE, MAX_PASSES, LIST_ONLY, FORCE)
except StopRun as e:
    result = None
    print("⚠️ ", e)

## Look at the result
The archive index, and the preview of the newest pass per area (ships are bright dots).
Open sea should be dark grey; a median VV of roughly −15 to −25 dB is normal.

In [ ]:
from IPython.display import Image as ShowImage, display

if result:
    index = os.path.join(result["data_dir"], "index.csv")
    if os.path.exists(index):
        try:
            import pandas as pd
            display(pd.read_csv(index))
        except ImportError:
            print(open(index).read())
        newest = {}
        with open(index, newline="") as f:
            for r in csv.DictReader(f):
                newest[r["area"]] = r
        for area, r in newest.items():
            ql = os.path.join(result["data_dir"], r["folder"], "quicklook.png")
            if os.path.exists(ql):
                print(f"{area}  {r['acquired_utc']}")
                display(ShowImage(filename=ql, width=700))